<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/cross_section_momentum/ON%20ENTIRE%20EQUITY/cross_sectional_momentum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#from google.colab import drive
#drive.mount('/content/drive')

In [4]:
import pandas as pd
import duckdb
import numpy as np

In [5]:
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/equities_combined.parquet")



In [6]:
df['TIMESTAMP'].max()

Timestamp('2026-03-06 00:00:00')

In [7]:
# Convert date column
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='%Y-%m-%d', errors='coerce')

# Remove bad rows
df = df.dropna(subset=['TIMESTAMP'])

# Remove duplicate symbol-date pairs
df = df.drop_duplicates(subset=['SYMBOL', 'TIMESTAMP'])

# Sort properly
df = df.sort_values(['SYMBOL', 'TIMESTAMP'])

# 6-month momentum (approx 126 trading days)
lookback = 126
df['mom_6m'] = df.groupby('SYMBOL')['CLOSE'].pct_change(lookback)

# Remove rows where momentum cannot be calculated
df = df.dropna(subset=['mom_6m'])

# Cross-sectional ranking each day
df['percentile'] = df.groupby('TIMESTAMP')['mom_6m'].rank(pct=True)

# Assign momentum buckets
df['momentum_group'] = 'middle'
df.loc[df['percentile'] >= 0.9, 'momentum_group'] = 'winner'
df.loc[df['percentile'] <= 0.1, 'momentum_group'] = 'loser'

# Winners and losers portfolios
long_portfolio = df[df['momentum_group'] == 'winner']
short_portfolio = df[df['momentum_group'] == 'loser']

# Display sample
print(df[['TIMESTAMP','SYMBOL','CLOSE','mom_6m','percentile','momentum_group']].head())

     TIMESTAMP      SYMBOL    CLOSE    mom_6m  percentile momentum_group
141 2026-02-23    0MOFSL26  1158.00  0.216387    0.891230         middle
142 2026-03-05    0MOFSL26  1161.89  0.162006    0.882752         middle
143 2026-03-06    0MOFSL26  1159.99  0.160106    0.883768         middle
535 2025-07-02  1003IIFL29   999.20  0.052998    0.730410         middle
536 2025-07-04  1003IIFL29   995.00 -0.014851    0.582374         middle


In [8]:
latest_date = df['TIMESTAMP'].max()

In [9]:
formation_date = '2026-03-06'

winners_today = df[
    (df['momentum_group'] == 'winner') &
    (df['TIMESTAMP'] == formation_date)


]

print(winners_today[['TIMESTAMP','SYMBOL','CLOSE','mom_6m','percentile','TOTTRDVAL']])

         TIMESTAMP      SYMBOL    CLOSE    mom_6m  percentile     TOTTRDVAL
162811  2026-03-06   ACCENTMIC   343.00  0.211372    0.902067  4.944975e+06
170786  2026-03-06     ACUTAAS  2250.70  0.600669    0.977296  8.207375e+08
171427  2026-03-06  ADANIENSOL   992.50  0.300701    0.925449  1.521960e+09
192973  2026-03-06    AEROFLEX   220.24  0.261253    0.916300  2.414017e+08
194778  2026-03-06      AETHER  1011.95  0.356592    0.937309  7.997529e+08
...            ...         ...      ...       ...         ...           ...
3640851 2026-03-06     VISAMAN   129.00  0.595547    0.976279  7.373500e+05
3656007 2026-03-06    VMARCIND   698.95  0.471009    0.960352  3.580496e+07
3671429 2026-03-06         VTL   538.90  0.307375    0.926804  1.025771e+08
3689047 2026-03-06      WELINV  1266.00  0.361583    0.939681  6.440532e+05
3718940 2026-03-06     XELPMOC   122.19  0.215216    0.902406  4.721182e+05

[296 rows x 6 columns]


In [10]:
# Liquidity filter
filtered = winners_today[df["TOTTRDQTY"] * df["TOTALTRADES"] > 500000]

# Sort by momentum
top50 = (
    filtered
    .sort_values("mom_6m", ascending=False)
    .head(50)
)

print(top50[["SYMBOL","TIMESTAMP","mom_6m","TOTTRDQTY","TOTALTRADES"]])

             SYMBOL  TIMESTAMP    mom_6m  TOTTRDQTY  TOTALTRADES
3334373        TAKE 2026-03-06  2.214953     299763         1440
2231732    MTARTECH 2026-03-06  1.627623     858506        91155
2856075    SALSTEEL 2026-03-06  1.491676       8364           65
1261999        GSTL 2026-03-06  1.349593      31000           18
1367928  HINDCOPPER 2026-03-06  1.349033    6695386        76986
2029488   MAHASTEEL 2026-03-06  1.269325      22037          949
327080        ARFIN 2026-03-06  1.236559    1161511         3824
778576        CUPID 2026-03-06  1.221854    6371108        63135
897911         DPEL 2026-03-06  1.125275      42500           72
3370245     TATSILV 2026-03-06  1.098745   99651160       105655
3114151   SILVERADD 2026-03-06  1.096880     425231         8059
3116755  SILVERIETF 2026-03-06  1.096133    6661095        34398
3535811   UNIHEALTH 2026-03-06  1.095912      26000           23
3115552  SILVERCASE 2026-03-06  1.093600   13849714        14896
3112953      SILVER 2026-

/tmp/ipykernel_17528/1776291802.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered = winners_today[df["TOTTRDQTY"] * df["TOTALTRADES"] > 500000]
